In [1]:
import os
import sys
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv

sys.path.append('..')
from parser.split import build_train_test_split

load_dotenv('../.env')
DATABASE_URL = (
    f"postgresql+psycopg2://{os.getenv('POSTGRES_USER')}:{os.getenv('POSTGRES_PASSWORD')}"
    f"@{os.getenv('POSTGRES_HOST')}:{os.getenv('POSTGRES_PORT')}/{os.getenv('POSTGRES_DB')}"
)
engine = create_engine(DATABASE_URL)

ratings, train, held_out = build_train_test_split()
recommendation_logs = pd.read_sql("SELECT * FROM recommendation_logs", engine)
experiment_assignments = pd.read_sql("SELECT * FROM experiment_assignments", engine)

print(recommendation_logs.shape, experiment_assignments.shape)

(6120, 5) (610, 3)


In [2]:
group_model_map = {"control": "popularity", "treatment": "hybrid"}
experiment_assignments["expected_model"] = experiment_assignments["group_name"].map(group_model_map)

valid_logs = recommendation_logs.merge(
    experiment_assignments[["user_id", "group_name", "expected_model"]],
    on="user_id"
)
valid_logs = valid_logs[valid_logs["model_used"] == valid_logs["expected_model"]]

print(valid_logs.shape)
print(valid_logs.groupby("group_name")["user_id"].nunique())

(6100, 7)
group_name
control      293
treatment    315
Name: user_id, dtype: int64


maps each group to its correct model, merges the logs with each user's assignment, then filters out any row where the logged model doesn't match what that user's group is actually supposed to receive — cleanly discarding the stale test entries.

In [3]:
from scoring.evaluation import build_held_out_liked

held_out_liked = build_held_out_liked(held_out)

def is_hit(row_group):
    user_id = row_group.name
    liked_movies = held_out_liked.get(user_id, [])
    recommended_movies = row_group['movie_id'].tolist()
    return int(any(movie in recommended_movies for movie in liked_movies))

user_hits = valid_logs.groupby('user_id').apply(is_hit, include_groups=False)
user_hits = user_hits.reset_index()
user_hits.columns = ['user_id', 'hit']

user_hits = user_hits.merge(experiment_assignments[['user_id', 'group_name']], on='user_id')
print(user_hits.groupby('group_name')['hit'].mean())

group_name
control      0.293515
treatment    0.425397
Name: hit, dtype: float64


In [4]:
from scipy import stats

control_hits = user_hits[user_hits['group_name'] == 'control']['hit']
treatment_hits = user_hits[user_hits['group_name'] == 'treatment']['hit']

count = [treatment_hits.sum(), control_hits.sum()]
nobs = [len(treatment_hits), len(control_hits)]

from statsmodels.stats.proportion import proportions_ztest, proportion_confint
z_stat, p_value = proportions_ztest(count, nobs)

print(f"Control hit-rate: {control_hits.mean():.2%} (n={len(control_hits)})")
print(f"Treatment hit-rate: {treatment_hits.mean():.2%} (n={len(treatment_hits)})")
print(f"Lift: {treatment_hits.mean() - control_hits.mean():.2%} points ({(treatment_hits.mean()-control_hits.mean())/control_hits.mean():.1%} relative)")
print(f"Z-statistic: {z_stat:.3f}, p-value: {p_value:.5f}")

Control hit-rate: 29.35% (n=293)
Treatment hit-rate: 42.54% (n=315)
Lift: 13.19% points (44.9% relative)
Z-statistic: 3.381, p-value: 0.00072


In [5]:
from statsmodels.stats.proportion import proportion_confint

control_ci = proportion_confint(control_hits.sum(), len(control_hits), method='wilson')
treatment_ci = proportion_confint(treatment_hits.sum(), len(treatment_hits), method='wilson')

lift = treatment_hits.mean() - control_hits.mean()
se_diff = ((control_hits.mean()*(1-control_hits.mean())/len(control_hits)) + (treatment_hits.mean()*(1-treatment_hits.mean())/len(treatment_hits))) ** 0.5
lift_ci = (lift - 1.96*se_diff, lift + 1.96*se_diff)

print(f"Control 95% CI: {control_ci[0]:.2%} - {control_ci[1]:.2%}")
print(f"Treatment 95% CI: {treatment_ci[0]:.2%} - {treatment_ci[1]:.2%}")
print(f"Lift 95% CI: {lift_ci[0]:.2%} - {lift_ci[1]:.2%} points")

Control 95% CI: 24.43% - 34.81%
Treatment 95% CI: 37.20% - 48.06%
Lift 95% CI: 5.64% - 20.74% points


| Group | Hit-rate@10 | n | 95% CI |
|---|---|---|---|
| Control (popularity) | 29.35% | 293 | 24.43% – 34.81% |
| Treatment (hybrid) | 42.54% | 315 | 37.20% – 48.06% |

**Lift: 13.19 percentage points (44.9% relative). p = 0.00072. Lift 95% CI: 5.64% – 20.74%.**

The A/B test rejects H0: the hybrid model produces a statistically significant
(p = 0.00072) and practically significant improvement over the popularity baseline.
The lift (13.19 points, 44.9% relative) clears both the pre-registered MDE (4.67 points,
15% relative lift) and the achievable-at-80%-power threshold (10.9 points) — even at the
conservative low end of the confidence interval (5.64 points), the result still clears
the MDE. This confirms the project's core hypothesis: personalized recommendations
(hybrid CF + content-based) drive meaningfully more engagement than a popularity-based
recommender.

**Known limitations**
- **Counterfactual bias**: this is an offline test against historical held-out data, not
  live user reactions to what was actually shown — a known limitation of offline recsys
  evaluation, not a live randomized experiment.
- **Small, fixed sample size** (~300 users/group): limits statistical power for smaller
  effects; this test is well-powered for large effects like the one observed, but would
  be unreliable for detecting a smaller true lift

In [6]:
import requests
response = requests.get(
    "https://api.themoviedb.org/3/movie/603",
    params={"api_key": "58e981b7b21df7cd609196c63eeab163"}
)
print(response.status_code)
print(response.json().get("title"), response.json().get("poster_path"))

200
The Matrix /dXNAPwY7VrqMAo51EKhhCJfaGb5.jpg
